In [1]:
import joblib
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

## load pkl file 

In [2]:
chunks = joblib.load("chunks.pkl")
index = faiss.read_index("faiss_index.index")

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## using the groq api key

In [ ]:
client = Groq(api_key="") # here using api key 

## function for question embedding, retrival, prompt and response

In [5]:
def answer_question(question, k = 3):
    question_embedding = embedding_model.encode([question], convert_to_numpy=True)

    faiss.normalize_L2(question_embedding)

    distances, indices = index.search(question_embedding, k=k)

    retrieved = [chunks[i] for i in indices[0] if i != -1]
    if not retrieved:
        return "I couldn't find anything relevant in the document to answer that."

    context = "\n\n".join(retrieved)

    prompt = f"""
                You are a Python tutor.

                Answer only using the context.

                Context:
                {context}

                Question:
                {question}

                Answer:
            """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


## ask question from user

In [8]:
question = input("Ask Question: ")
question

'what is single ton class with example'

## get response from model

In [9]:
print(answer_question(question))

A singleton class is a class that can have only one object (an instance of the class) at a time. It is a class that can have only one instance and provides a global point of access to that instance.

Here's an example of a singleton class in Python:

```python
class Singleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(Singleton, cls).__new__(cls)
        return cls._instance

# Create two instances of the Singleton class
s1 = Singleton()
s2 = Singleton()

# Check if both instances are the same
print(s1 is s2)  # Output: True
```

In this example, the `Singleton` class overrides the `__new__` method to ensure that only one instance of the class is created. The `_instance` attribute is used to store the single instance of the class. If an instance already exists, the `__new__` method returns the existing instance; otherwise, it creates a new instance and stores it in the `_instance` attribute.

When you create two inst